In [ ]:
# Comprehensive lexical analysis with PLY
import ply.lex as lex
import tempfile
import os
from typing import List, Dict, Any

# Create a temporary module to work around Jupyter's __main__ module issues
temp_dir = tempfile.mkdtemp()
lexer_module_path = os.path.join(temp_dir, 'calculator_lexer.py')

# Define a complete lexer module as a string
lexer_module_content = '''
import ply.lex as lex

# Token definitions for a simple calculator
tokens = (
    'NUMBER',
    'PLUS',
    'MINUS',
    'TIMES',
    'DIVIDE',
    'LPAREN',
    'RPAREN',
    'ID',
    'ASSIGN',
)

# Token rules
t_PLUS    = r'\\+'
t_MINUS   = r'-'
t_TIMES   = r'\\*'
t_DIVIDE  = r'/'
t_LPAREN  = r'\\('
t_RPAREN  = r'\\)'
t_ASSIGN  = r'='

# Ignored characters (spaces and tabs)
t_ignore = ' \\t'

def t_NUMBER(t):
    r'\\d+'
    t.value = int(t.value)
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    return t

def t_newline(t):
    r'\\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    print(f"Illegal character '{t.value[0]}'")
    t.lexer.skip(1)

# Build the lexer
lexer = lex.lex()
'''

# Write the lexer module to a file
with open(lexer_module_path, 'w') as f:
    f.write(lexer_module_content)

# Import the module dynamically
import sys
sys.path.insert(0, temp_dir)
import calculator_lexer

print("PLY Lexer Examples:")
print(f"Available tokens: {calculator_lexer.tokens}")
assert calculator_lexer.tokens == (
    'NUMBER', 'PLUS', 'MINUS', 'TIMES', 'DIVIDE', 'LPAREN', 'RPAREN', 'ID', 'ASSIGN',
)

def tokenize(text):
    calculator_lexer.lexer.input(text)
    return [(tok.type, tok.value) for tok in iter(calculator_lexer.lexer.token, None)]

# Test the lexer with various inputs
test_inputs = [
    "123 + 456",
    "x = 10 * (y + 5)",
    "result = a + b - c / d",
    "variable123 = (100 + 200) * 3"
]

print(f"\nTokenization examples:")
tokenized = {}
for i, test_input in enumerate(test_inputs, 1):
    print(f"\nExample {i}: '{test_input}'")
    tokenized[test_input] = tokenize(test_input)
    print(f"  Tokens: {tokenized[test_input]}")

# NUMBER tokens carry int values; operators and names carry their text
assert tokenized["123 + 456"] == [('NUMBER', 123), ('PLUS', '+'), ('NUMBER', 456)]
assert [t for t, _ in tokenized["x = 10 * (y + 5)"]] == [
    'ID', 'ASSIGN', 'NUMBER', 'TIMES', 'LPAREN', 'ID', 'PLUS', 'NUMBER', 'RPAREN',
]
assert [t for t, _ in tokenized["result = a + b - c / d"]] == [
    'ID', 'ASSIGN', 'ID', 'PLUS', 'ID', 'MINUS', 'ID', 'DIVIDE', 'ID',
]
assert tokenized["variable123 = (100 + 200) * 3"][0] == ('ID', 'variable123')
assert [v for t, v in tokenized["variable123 = (100 + 200) * 3"] if t == 'NUMBER'] == [100, 200, 3]

# Newlines are not tokens, but they advance the line number
calculator_lexer.lexer.lineno = 1
calculator_lexer.lexer.input("a\nb\n\nc")
line_numbers = [(tok.value, tok.lineno) for tok in iter(calculator_lexer.lexer.token, None)]
print(f"\nLine numbers: {line_numbers}")
assert line_numbers == [('a', 1), ('b', 2), ('c', 4)]

# An illegal character is reported by t_error and skipped
import contextlib, io
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    skipped = tokenize("1 $ 2")
print(f"Tokens around an illegal character: {skipped}")
assert skipped == [('NUMBER', 1), ('NUMBER', 2)]
assert "Illegal character '$'" in captured.getvalue()

# Clean up
sys.path.remove(temp_dir)
import shutil
shutil.rmtree(temp_dir)



In [ ]:
# Advanced lexer with states and more complex tokens
import ply.lex as lex
import tempfile
import os

# Create another temporary module for advanced lexer
temp_dir = tempfile.mkdtemp()
advanced_lexer_path = os.path.join(temp_dir, 'advanced_lexer.py')

advanced_lexer_content = '''
import ply.lex as lex

# Define tokens for a more complex language
tokens = (
    'NUMBER',
    'FLOAT', 
    'STRING',
    'ID',
    'IF',
    'ELSE',
    'WHILE',
    'FOR',
    'FUNCTION',
    'RETURN',
    'ASSIGN',
    'EQ',
    'NE',
    'LT',
    'LE',
    'GT',
    'GE',
    'PLUS',
    'MINUS',
    'TIMES',
    'DIVIDE',
    'LPAREN',
    'RPAREN',
    'LBRACE',
    'RBRACE',
    'SEMICOLON',
    'COMMA',
)

# Reserved words
reserved = {
    'if': 'IF',
    'else': 'ELSE', 
    'while': 'WHILE',
    'for': 'FOR',
    'function': 'FUNCTION',
    'return': 'RETURN',
}

# Token rules
t_ASSIGN   = r'='
t_EQ       = r'=='
t_NE       = r'!='
t_LT       = r'<'
t_LE       = r'<='
t_GT       = r'>'
t_GE       = r'>='
t_PLUS     = r'\\+'
t_MINUS    = r'-'
t_TIMES    = r'\\*'
t_DIVIDE   = r'/'
t_LPAREN   = r'\\('
t_RPAREN   = r'\\)'
t_LBRACE   = r'\\{'
t_RBRACE   = r'\\}'
t_SEMICOLON = r';'
t_COMMA    = r','

# Ignored characters
t_ignore = ' \\t'

def t_FLOAT(t):
    r'\\d+\\.\\d+'
    t.value = float(t.value)
    return t

def t_NUMBER(t):
    r'\\d+'
    t.value = int(t.value)
    return t

def t_STRING(t):
    r'"([^"\\\\]|\\\\.)*"'
    t.value = t.value[1:-1]  # Remove quotes
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    t.type = reserved.get(t.value, 'ID')
    return t

def t_newline(t):
    r'\\n+'
    t.lexer.lineno += len(t.value)

def t_COMMENT(t):
    r'//.*'
    pass  # No return value. Token discarded

def t_error(t):
    print(f"Illegal character '{t.value[0]}' at line {t.lexer.lineno}")
    t.lexer.skip(1)

# Build the lexer
lexer = lex.lex()
'''

# Write and import the advanced lexer
with open(advanced_lexer_path, 'w') as f:
    f.write(advanced_lexer_content)

import sys
sys.path.insert(0, temp_dir)
import advanced_lexer

print("Advanced Lexer Examples:")
print(f"Advanced lexer created with {len(advanced_lexer.tokens)} token types")
assert len(advanced_lexer.tokens) == 27

# Test with more complex code samples
complex_inputs = [
    'if (x == 10) { return "hello"; }',
    'function calculate(a, b) { return a + b * 3.14; }',
    'while (i < 100) { i = i + 1; // increment }',
    'for (i = 0; i <= n; i = i + 1) { sum = sum + i; }'
]

print(f"\nComplex tokenization examples:")
lexed = []
for i, code in enumerate(complex_inputs, 1):
    print(f"\nExample {i}: {code}")
    advanced_lexer.lexer.input(code)
    tokens_list = [(tok.type, tok.value) for tok in iter(advanced_lexer.lexer.token, None)]
    lexed.append(tokens_list)
    print(f"  Tokens: {' '.join(f'{t}({v!r})' for t, v in tokens_list)}")

# Reserved words become keyword tokens, '==' wins over '=', and the string loses its quotes
assert lexed[0] == [
    ('IF', 'if'), ('LPAREN', '('), ('ID', 'x'), ('EQ', '=='), ('NUMBER', 10), ('RPAREN', ')'),
    ('LBRACE', '{'), ('RETURN', 'return'), ('STRING', 'hello'), ('SEMICOLON', ';'), ('RBRACE', '}'),
]
# A decimal literal is a FLOAT token with a float value
assert lexed[1][0] == ('FUNCTION', 'function')
assert ('FLOAT', 3.14) in lexed[1] and isinstance(dict(lexed[1])['FLOAT'], float)
assert [t for t, _ in lexed[1]].count('ID') == 5
# The // comment runs to the end of the line, so the closing brace is discarded with it
assert lexed[2][-1] == ('SEMICOLON', ';')
assert 'RBRACE' not in [t for t, _ in lexed[2]]
# '<=' is one LE token, not LT followed by ASSIGN
assert ('LE', '<=') in lexed[3] and 'LT' not in [t for t, _ in lexed[3]]
assert [t for t, _ in lexed[3]].count('SEMICOLON') == 3

# Clean up
sys.path.remove(temp_dir)
import shutil
shutil.rmtree(temp_dir)


In [ ]:
# Comprehensive parser creation with PLY yacc
import ply.yacc as yacc
import ply.lex as lex
import tempfile
import os

# Create a complete calculator parser module
temp_dir = tempfile.mkdtemp()
parser_module_path = os.path.join(temp_dir, 'calculator_parser.py')

parser_module_content = '''
import ply.lex as lex
import ply.yacc as yacc

# Lexer tokens (reuse from previous examples)
tokens = (
    'NUMBER',
    'PLUS',
    'MINUS',
    'TIMES',
    'DIVIDE',
    'LPAREN',
    'RPAREN',
    'ID',
    'ASSIGN',
)

# Token rules
t_PLUS    = r'\\+'
t_MINUS   = r'-'
t_TIMES   = r'\\*'
t_DIVIDE  = r'/'
t_LPAREN  = r'\\('
t_RPAREN  = r'\\)'
t_ASSIGN  = r'='
t_ignore  = ' \\t'

def t_NUMBER(t):
    r'\\d+'
    t.value = int(t.value)
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    return t

def t_newline(t):
    r'\\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    print(f"Illegal character '{t.value[0]}'")
    t.lexer.skip(1)

# Build lexer
lexer = lex.lex()

# Operator precedence and associativity
precedence = (
    ('left', 'PLUS', 'MINUS'),
    ('left', 'TIMES', 'DIVIDE'),
    ('right', 'UMINUS'),  # Unary minus operator
)

# Grammar rules
def p_statement_assign(p):
    """statement : ID ASSIGN expression"""
    names[p[1]] = p[3]
    p[0] = p[3]

def p_statement_expr(p):
    """statement : expression"""
    p[0] = p[1]

def p_expression_binop(p):
    """expression : expression PLUS expression
                  | expression MINUS expression
                  | expression TIMES expression
                  | expression DIVIDE expression"""
    if p[2] == '+':
        p[0] = p[1] + p[3]
    elif p[2] == '-':
        p[0] = p[1] - p[3]
    elif p[2] == '*':
        p[0] = p[1] * p[3]
    elif p[2] == '/':
        if p[3] == 0:
            print("Division by zero!")
            p[0] = 0
        else:
            p[0] = p[1] / p[3]

def p_expression_uminus(p):
    """expression : MINUS expression %prec UMINUS"""
    p[0] = -p[2]

def p_expression_group(p):
    """expression : LPAREN expression RPAREN"""
    p[0] = p[2]

def p_expression_number(p):
    """expression : NUMBER"""
    p[0] = p[1]

def p_expression_name(p):
    """expression : ID"""
    try:
        p[0] = names[p[1]]
    except LookupError:
        print(f"Undefined name '{p[1]}'")
        p[0] = 0

def p_error(p):
    if p:
        print(f"Syntax error at token {p.type} ('{p.value}')")
    else:
        print("Syntax error at EOF")

# Variable storage
names = {}

# Build parser
parser = yacc.yacc()

def evaluate(expression):
    """Evaluate a mathematical expression."""
    return parser.parse(expression, lexer=lexer)
'''

# Write the parser module
with open(parser_module_path, 'w') as f:
    f.write(parser_module_content)

# Import the parser module
import sys
sys.path.insert(0, temp_dir)
import calculator_parser

print("PLY Parser Examples:")

# Test expressions: precedence, associativity, unary minus and assignments
expected_results = {
    "2 + 3 * 4": 14,
    "(2 + 3) * 4": 20,
    "10 - 5 + 2": 7,
    "20 / 4 + 1": 6.0,
    "-5 + 10": 5,
    "x = 42": 42,
    "y = x + 8": 50,
    "result = (x + y) * 2": 184,
}

print(f"\nParsing and evaluation examples:")
for i, expr in enumerate(expected_results, 1):
    result = calculator_parser.evaluate(expr)
    print(f"  {i}. {expr:20} = {result}")
    assert result == expected_results[expr], (expr, result)

# Show variable storage
print(f"\nVariables stored: {calculator_parser.names}")
assert calculator_parser.names == {'x': 42, 'y': 50, 'result': 184}

# Error handling: the grammar's p_error and the actions report problems instead of raising
import contextlib, io
print(f"\nError handling examples:")
error_cases = {
    "2 + + 3": "Syntax error at token PLUS",
    "5 / 0": "Division by zero!",
    "undefined_var + 1": "Undefined name 'undefined_var'",
}
error_results = {}
for expr, message in error_cases.items():
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        error_results[expr] = calculator_parser.evaluate(expr)
    print(f"  {expr}: {captured.getvalue().strip()} -> {error_results[expr]}")
    assert message in captured.getvalue(), (expr, captured.getvalue())

# Division by zero yields 0, and an undefined name evaluates as 0
assert error_results["5 / 0"] == 0
assert error_results["undefined_var + 1"] == 1

# Clean up
sys.path.remove(temp_dir)
import shutil
shutil.rmtree(temp_dir)



In [ ]:
# Advanced parsing features and AST generation
import ply.lex as lex
import ply.yacc as yacc
import tempfile
import os

# Create an AST (Abstract Syntax Tree) parser
temp_dir = tempfile.mkdtemp()
ast_parser_path = os.path.join(temp_dir, 'ast_parser.py')

ast_parser_content = '''
import ply.lex as lex
import ply.yacc as yacc

# AST Node classes
class ASTNode:
    pass

class BinaryOp(ASTNode):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right
    
    def __str__(self):
        return f"({self.left} {self.op} {self.right})"

class UnaryOp(ASTNode):
    def __init__(self, op, expr):
        self.op = op
        self.expr = expr
    
    def __str__(self):
        return f"({self.op}{self.expr})"

class Number(ASTNode):
    def __init__(self, value):
        self.value = value
    
    def __str__(self):
        return str(self.value)

class Identifier(ASTNode):
    def __init__(self, name):
        self.name = name
    
    def __str__(self):
        return self.name

class Assignment(ASTNode):
    def __init__(self, name, expr):
        self.name = name
        self.expr = expr
    
    def __str__(self):
        return f"{self.name} = {self.expr}"

# Lexer tokens
tokens = (
    'NUMBER', 'PLUS', 'MINUS', 'TIMES', 'DIVIDE',
    'LPAREN', 'RPAREN', 'ID', 'ASSIGN'
)

t_PLUS = r'\\+'
t_MINUS = r'-'
t_TIMES = r'\\*'
t_DIVIDE = r'/'
t_LPAREN = r'\\('
t_RPAREN = r'\\)'
t_ASSIGN = r'='
t_ignore = ' \\t'

def t_NUMBER(t):
    r'\\d+'
    t.value = int(t.value)
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    return t

def t_newline(t):
    r'\\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    print(f"Illegal character '{t.value[0]}'")
    t.lexer.skip(1)

lexer = lex.lex()

# Parser rules for AST generation
precedence = (
    ('left', 'PLUS', 'MINUS'),
    ('left', 'TIMES', 'DIVIDE'),
    ('right', 'UMINUS'),
)

def p_statement_assign(p):
    """statement : ID ASSIGN expression"""
    p[0] = Assignment(p[1], p[3])

def p_statement_expr(p):
    """statement : expression"""
    p[0] = p[1]

def p_expression_binop(p):
    """expression : expression PLUS expression
                  | expression MINUS expression
                  | expression TIMES expression
                  | expression DIVIDE expression"""
    p[0] = BinaryOp(p[1], p[2], p[3])

def p_expression_uminus(p):
    """expression : MINUS expression %prec UMINUS"""
    p[0] = UnaryOp(p[1], p[2])

def p_expression_group(p):
    """expression : LPAREN expression RPAREN"""
    p[0] = p[2]

def p_expression_number(p):
    """expression : NUMBER"""
    p[0] = Number(p[1])

def p_expression_name(p):
    """expression : ID"""
    p[0] = Identifier(p[1])

def p_error(p):
    if p:
        print(f"Syntax error at token {p.type}")
    else:
        print("Syntax error at EOF")

parser = yacc.yacc()

def parse_to_ast(expression):
    """Parse expression and return AST."""
    return parser.parse(expression, lexer=lexer)

# AST Evaluator
class ASTEvaluator:
    def __init__(self):
        self.variables = {}
    
    def evaluate(self, node):
        if isinstance(node, Number):
            return node.value
        elif isinstance(node, Identifier):
            if node.name in self.variables:
                return self.variables[node.name]
            else:
                raise NameError(f"Undefined variable: {node.name}")
        elif isinstance(node, BinaryOp):
            left = self.evaluate(node.left)
            right = self.evaluate(node.right)
            if node.op == '+':
                return left + right
            elif node.op == '-':
                return left - right
            elif node.op == '*':
                return left * right
            elif node.op == '/':
                if right == 0:
                    raise ZeroDivisionError("Division by zero")
                return left / right
        elif isinstance(node, UnaryOp):
            if node.op == '-':
                return -self.evaluate(node.expr)
        elif isinstance(node, Assignment):
            value = self.evaluate(node.expr)
            self.variables[node.name] = value
            return value
        
        raise ValueError(f"Unknown node type: {type(node)}")

evaluator = ASTEvaluator()
'''

# Write and import AST parser
with open(ast_parser_path, 'w') as f:
    f.write(ast_parser_content)

import sys
sys.path.insert(0, temp_dir)
import ast_parser

print("Advanced Parser Features - AST Generation:")

# Parse each expression to a tree, print it, then evaluate it with fixed variable values
expected = {
    "2 + 3 * 4": ("(2 + (3 * 4))", 14),
    "(x + y) * z": ("((x + y) * z)", 30),
    "-5 + 10": ("((-5) + 10)", 5),
    "result = a + b * c": ("result = (a + (b * c))", 27),
}

print(f"\nAST Generation Examples:")
ast_parser.evaluator.variables = {'x': 10, 'y': 5, 'z': 2, 'a': 3, 'b': 4, 'c': 6}
for i, expr in enumerate(expected, 1):
    print(f"\nExample {i}: {expr}")
    tree = ast_parser.parse_to_ast(expr)
    result = ast_parser.evaluator.evaluate(tree)
    print(f"  AST: {tree}")
    print(f"  Evaluated: {result}")
    assert (str(tree), result) == expected[expr], (expr, str(tree), result)

# Precedence shapes the tree: '*' binds tighter than '+', so it sits below it
tree = ast_parser.parse_to_ast("2 + 3 * 4")
assert isinstance(tree, ast_parser.BinaryOp) and tree.op == '+'
assert isinstance(tree.left, ast_parser.Number) and tree.left.value == 2
assert isinstance(tree.right, ast_parser.BinaryOp) and tree.right.op == '*'
# Subtraction is left-associative: 10 - 5 - 2 is (10 - 5) - 2
tree = ast_parser.parse_to_ast("10 - 5 - 2")
assert str(tree) == "((10 - 5) - 2)" and ast_parser.evaluator.evaluate(tree) == 3
assert isinstance(ast_parser.parse_to_ast("-x"), ast_parser.UnaryOp)

# The assignment was recorded by the evaluator
print(f"\nFinal variables: {ast_parser.evaluator.variables}")
assert ast_parser.evaluator.variables['result'] == 27

# Clean up
sys.path.remove(temp_dir)
import shutil
shutil.rmtree(temp_dir)


In [ ]:
# Real-world parser example: Simple programming language
import ply.lex as lex
import ply.yacc as yacc
import tempfile
import os

# Create a mini programming language parser
temp_dir = tempfile.mkdtemp()
language_parser_path = os.path.join(temp_dir, 'mini_language.py')

language_parser_content = '''
import ply.lex as lex
import ply.yacc as yacc

# Token definitions for a mini programming language
tokens = (
    'NUMBER', 'STRING', 'ID',
    'IF', 'ELSE', 'WHILE', 'FOR', 'FUNCTION', 'RETURN', 'PRINT',
    'ASSIGN', 'EQ', 'NE', 'LT', 'LE', 'GT', 'GE',
    'PLUS', 'MINUS', 'TIMES', 'DIVIDE',
    'LPAREN', 'RPAREN', 'LBRACE', 'RBRACE',
    'SEMICOLON', 'COMMA',
)

# Reserved words
reserved = {
    'if': 'IF',
    'else': 'ELSE',
    'while': 'WHILE', 
    'for': 'FOR',
    'function': 'FUNCTION',
    'return': 'RETURN',
    'print': 'PRINT',
}

# Token rules
t_ASSIGN = r'='
t_EQ = r'=='
t_NE = r'!='
t_LT = r'<'
t_LE = r'<='
t_GT = r'>'
t_GE = r'>='
t_PLUS = r'\\+'
t_MINUS = r'-'
t_TIMES = r'\\*'
t_DIVIDE = r'/'
t_LPAREN = r'\\('
t_RPAREN = r'\\)'
t_LBRACE = r'\\{'
t_RBRACE = r'\\}'
t_SEMICOLON = r';'
t_COMMA = r','
t_ignore = ' \\t'

def t_NUMBER(t):
    r'\\d+'
    t.value = int(t.value)
    return t

def t_STRING(t):
    r'"([^"\\\\]|\\\\.)*"'
    t.value = t.value[1:-1]  # Remove quotes
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    t.type = reserved.get(t.value, 'ID')
    return t

def t_newline(t):
    r'\\n+'
    t.lexer.lineno += len(t.value)

def t_COMMENT(t):
    r'//.*'
    pass

def t_error(t):
    print(f"Illegal character '{t.value[0]}' at line {t.lexer.lineno}")
    t.lexer.skip(1)

lexer = lex.lex()

# AST Node classes for the programming language
class Program:
    def __init__(self, statements):
        self.statements = statements
    
    def __str__(self):
        return "\\n".join(str(stmt) for stmt in self.statements)

class IfStatement:
    def __init__(self, condition, then_stmt, else_stmt=None):
        self.condition = condition
        self.then_stmt = then_stmt
        self.else_stmt = else_stmt
    
    def __str__(self):
        result = f"if ({self.condition}) {self.then_stmt}"
        if self.else_stmt:
            result += f" else {self.else_stmt}"
        return result

class WhileStatement:
    def __init__(self, condition, body):
        self.condition = condition
        self.body = body
    
    def __str__(self):
        return f"while ({self.condition}) {self.body}"

class PrintStatement:
    def __init__(self, expr):
        self.expr = expr
    
    def __str__(self):
        return f"print({self.expr})"

class Block:
    def __init__(self, statements):
        self.statements = statements
    
    def __str__(self):
        return "{ " + "; ".join(str(stmt) for stmt in self.statements) + " }"

class Assignment:
    def __init__(self, name, expr):
        self.name = name
        self.expr = expr
    
    def __str__(self):
        return f"{self.name} = {self.expr}"

class BinaryOp:
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right
    
    def __str__(self):
        return f"({self.left} {self.op} {self.right})"

class Number:
    def __init__(self, value):
        self.value = value
    
    def __str__(self):
        return str(self.value)

class String:
    def __init__(self, value):
        self.value = value
    
    def __str__(self):
        return f'"{self.value}"'

class Identifier:
    def __init__(self, name):
        self.name = name
    
    def __str__(self):
        return self.name

# Grammar rules
precedence = (
    ('left', 'EQ', 'NE'),
    ('left', 'LT', 'LE', 'GT', 'GE'),
    ('left', 'PLUS', 'MINUS'),
    ('left', 'TIMES', 'DIVIDE'),
)

def p_program(p):
    """program : statement_list"""
    p[0] = Program(p[1])

def p_statement_list(p):
    """statement_list : statement_list statement
                     | statement"""
    if len(p) == 2:
        p[0] = [p[1]]
    else:
        p[1].append(p[2])
        p[0] = p[1]

def p_statement(p):
    """statement : if_statement
                | while_statement
                | print_statement SEMICOLON
                | assignment SEMICOLON
                | block"""
    p[0] = p[1]

def p_if_statement(p):
    """if_statement : IF LPAREN expression RPAREN statement
                   | IF LPAREN expression RPAREN statement ELSE statement"""
    if len(p) == 6:
        p[0] = IfStatement(p[3], p[5])
    else:
        p[0] = IfStatement(p[3], p[5], p[7])

def p_while_statement(p):
    """while_statement : WHILE LPAREN expression RPAREN statement"""
    p[0] = WhileStatement(p[3], p[5])

def p_print_statement(p):
    """print_statement : PRINT LPAREN expression RPAREN"""
    p[0] = PrintStatement(p[3])

def p_assignment(p):
    """assignment : ID ASSIGN expression"""
    p[0] = Assignment(p[1], p[3])

def p_block(p):
    """block : LBRACE statement_list RBRACE"""
    p[0] = Block(p[2])

def p_expression_binop(p):
    """expression : expression PLUS expression
                  | expression MINUS expression
                  | expression TIMES expression
                  | expression DIVIDE expression
                  | expression EQ expression
                  | expression NE expression
                  | expression LT expression
                  | expression LE expression
                  | expression GT expression
                  | expression GE expression"""
    p[0] = BinaryOp(p[1], p[2], p[3])

def p_expression_group(p):
    """expression : LPAREN expression RPAREN"""
    p[0] = p[2]

def p_expression_number(p):
    """expression : NUMBER"""
    p[0] = Number(p[1])

def p_expression_string(p):
    """expression : STRING"""
    p[0] = String(p[1])

def p_expression_name(p):
    """expression : ID"""
    p[0] = Identifier(p[1])

def p_error(p):
    if p:
        print(f"Syntax error at token {p.type} ('{p.value}') at line {p.lineno}")
    else:
        print("Syntax error at EOF")

parser = yacc.yacc()

def parse_program(code):
    """Parse a mini program and return AST."""
    return parser.parse(code, lexer=lexer)
'''

# Write and import the language parser
with open(language_parser_path, 'w') as f:
    f.write(language_parser_content)

import sys
sys.path.insert(0, temp_dir)
import mini_language

print("Real-World Parser Example: Mini Programming Language")
print("Lexer and parser for assignments, if/else, while loops and print statements")

# Test programs
test_programs = [
    '''
    x = 10;
    y = 20;
    print(x + y);
    ''',
    
    '''
    if (x > 5) {
        print("x is greater than 5");
    } else {
        print("x is not greater than 5");
    }
    ''',
    
    '''
    i = 0;
    while (i < 3) {
        print(i);
        i = i + 1;
    }
    ''',
    
    '''
    name = "World";
    print("Hello " + name);
    '''
]

print(f"\nParsing complete programs:")
parsed = []
for i, program in enumerate(test_programs, 1):
    print(f"\nProgram {i}:")
    print(f"Source:{program}")
    tree = mini_language.parse_program(program.strip())
    parsed.append(tree)
    print(f"Parsed AST: {tree}")

assert all(isinstance(tree, mini_language.Program) for tree in parsed)
assert str(parsed[0]) == "x = 10\ny = 20\nprint((x + y))"
assert [type(s).__name__ for s in parsed[0].statements] == ['Assignment', 'Assignment', 'PrintStatement']

if_stmt = parsed[1].statements[0]
assert isinstance(if_stmt, mini_language.IfStatement)
assert str(if_stmt.condition) == "(x > 5)"
assert isinstance(if_stmt.then_stmt, mini_language.Block) and isinstance(if_stmt.else_stmt, mini_language.Block)
assert str(if_stmt.else_stmt) == '{ print("x is not greater than 5") }'

while_stmt = parsed[2].statements[1]
assert isinstance(while_stmt, mini_language.WhileStatement)
assert str(while_stmt) == "while ((i < 3)) { print(i); i = (i + 1) }"
assert len(while_stmt.body.statements) == 2

assert str(parsed[3]) == 'name = "World"\nprint(("Hello " + name))'
assert isinstance(parsed[3].statements[0].expr, mini_language.String)

import importlib.metadata
import ply
print(f"\nPLY modules available:")
print(f"  - ply.lex: {ply.lex.__name__}")
print(f"  - ply.yacc: {ply.yacc.__name__}")
print(f"  - Version: {ply.__version__}")
assert ply.__version__ == importlib.metadata.version("ply")

# Clean up
sys.path.remove(temp_dir)
import shutil
shutil.rmtree(temp_dir)

